# Lead assignment (standalone notebook)

Self-contained version of `assign_leads.ipynb`: same flow and same result, but all the code
(connection, seller normalization, candidate selection and the assignment algorithm) lives in
this notebook, so it does not import anything from `utils/` and can run from any folder.

```
leads_clean  +  sales_people  ->  lead_assignments
```

It reads the new leads and the active sellers, assigns each lead to a seller based on
**language** and **content of interest**, and materializes the result in `lead_assignments`
(ClickHouse, `DB_NAME` database).

## 1. Parameters

The next cell is tagged `parameters`. Intelek injects `creds` (the secret object, a dict) in the first
cell of the notebook; plain papermill injects the parameters right after this cell. Either way the
injected values win. Locally `creds` stays `None` and the connection reads `.env`.

In [ ]:
try:
    creds
except NameError:
    creds = None   # local runs: read DB_* from .env
LEADS_TABLE = "leads_clean"
SELLERS_TABLE = "sales_people"
OUTPUT_TABLE = "lead_assignments"

FULL_REFRESH = False   # True: ignore previous assignments and rebuild the table
DRY_RUN = False        # True: compute but do not write

## 2. Imports and constants

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

# Keys of the Intelek secret object, and names of the variables in .env
CREDS_KEYS = ["DB_HOST", "DB_PORT", "DB_USER", "DB_PASSWORD", "DB_NAME"]
SECURE_PORTS = {8443}   # HTTPS, e.g. ClickHouse Cloud

# Columns of the raw sales_people table (already snake_case at the origin)
SELLER_COLUMNS = ["seller_id", "full_name", "email", "languages", "specialties",
                  "max_open_leads", "active"]

FULL_MATCH = "full_match"          # speaks the language AND is specialized in the content
LANGUAGE_ONLY = "language_only"    # speaks the language, but no specialist available
UNASSIGNED = "unassigned"          # nobody speaks the language, or everybody is full

OUTPUT_COLUMNS = ["lead_id", "seller_id", "match_type", "reason",
                  "preferred_language", "content_of_interest", "assigned_at", "run_id"]

## 3. Database helpers

Credentials, connection, and read / write of tables. A notebook does not know where its own file is,
so for local runs `.env` is looked up from the working directory: `.env`, `compute/.env` or `../.env`
(Jupyter starts in this notebook's folder, a scheduler may start in the repo root).

In [ ]:
def load_creds(creds=None) -> dict:
    """The injected creds dict as-is or, if None, the same variables from .env (local runs)."""
    if creds is not None:
        return creds
    env_file = next((p for p in [Path.cwd() / ".env", Path.cwd() / "compute" / ".env",
                                 Path.cwd().parent / ".env"] if p.is_file()), None)
    if env_file is None:
        raise FileNotFoundError(f"No creds injected and no .env found from {Path.cwd()}")
    load_dotenv(env_file, override=True)
    return {key: os.environ[key] for key in CREDS_KEYS}


def get_client(creds=None):
    """ClickHouse client; unqualified table names resolve to the DB_NAME database."""
    creds = load_creds(creds)
    port = int(creds.get("DB_PORT", 8123))
    return clickhouse_connect.get_client(
        host=creds["DB_HOST"], port=port, username=creds["DB_USER"],
        password=creds["DB_PASSWORD"], database=creds["DB_NAME"], secure=port in SECURE_PORTS)


def clickhouse_type(dtype) -> str:
    """pandas dtype -> Nullable ClickHouse type."""
    if pd.api.types.is_bool_dtype(dtype):
        base = "Bool"
    elif pd.api.types.is_integer_dtype(dtype):
        base = "Int64"
    elif pd.api.types.is_float_dtype(dtype):
        base = "Float64"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        base = "DateTime64(3, 'UTC')"   # naive timestamps, kept as-is
    else:
        base = "String"
    return f"Nullable({base})"


def table_exists(client, table: str) -> bool:
    return bool(client.command(f"EXISTS TABLE {table}"))


def read_table(client, table: str) -> pd.DataFrame:
    """Read a full table. Timestamps come back naive, as they were written."""
    df = client.query_df(f"SELECT * FROM {table}")
    for col in df.columns:
        if isinstance(df[col].dtype, pd.DatetimeTZDtype):
            df[col] = df[col].dt.tz_localize(None)
    return df


def write_table(df: pd.DataFrame, client, table: str, if_exists: str = "append") -> None:
    """Write df to `table`, creating it from the DataFrame columns if needed.
    if_exists: 'append' | 'replace' | 'fail' (same as pandas.to_sql)."""
    exists = table_exists(client, table)
    if exists and if_exists == "fail":
        raise ValueError(f"Table {table} already exists")
    if exists and if_exists == "replace":
        client.command(f"DROP TABLE {table}")
        exists = False
    if not exists:
        columns = ", ".join(f"`{col}` {clickhouse_type(dtype)}" for col, dtype in df.dtypes.items())
        client.command(f"CREATE TABLE {table} ({columns}) ENGINE = MergeTree ORDER BY tuple()")
    rows = df.astype(object).where(df.notna(), None).values.tolist()   # NaN/NaT -> NULL
    client.insert(table, rows, column_names=list(df.columns))

## 4. Sellers

Keep the seller columns, parse the `;`-separated lists into sets and cast types.

In [ ]:
def split_list(value) -> frozenset:
    """'es; CA ;en' -> frozenset({'es', 'ca', 'en'}). Empty/NULL -> empty set."""
    if value is None or pd.isna(value):
        return frozenset()
    return frozenset(v.strip().lower() for v in str(value).split(";") if v.strip())


def normalize_sellers(raw: pd.DataFrame, only_active: bool = True) -> pd.DataFrame:
    """One row per seller: seller_id, full_name, email, languages (set), specialties (set),
    max_open_leads (int), is_active (bool)."""
    sellers = raw[SELLER_COLUMNS].copy()
    sellers["languages"] = sellers["languages"].map(split_list)
    sellers["specialties"] = sellers["specialties"].map(split_list)
    sellers["max_open_leads"] = pd.to_numeric(sellers["max_open_leads"], errors="coerce").fillna(0).astype(int)
    sellers["is_active"] = sellers.pop("active").astype(str).str.strip().str.lower().eq("yes")
    if only_active:
        sellers = sellers[sellers["is_active"]]
    return sellers.sort_values("seller_id").reset_index(drop=True)

## 5. Assignment algorithm (pure pandas, no database)

Rules, lead by lead in order of arrival (`created_at`):
1. Prefer sellers who speak the lead's language **and** are specialized in its content (`full_match`).
2. Otherwise, any seller speaking the language (`language_only`).
3. Within each tier, the seller with the lowest load / capacity ratio that still has capacity wins
   (ties go to the lower `seller_id`). If nobody is available the lead stays `unassigned`.

In [ ]:
def candidate_tiers(language: str, content: str, sellers: pd.DataFrame) -> list:
    """Candidate sellers for a lead as [(match_type, sellers_df), ...], specialists first."""
    speaks = sellers["languages"].map(lambda langs: language in langs)
    specialist = sellers["specialties"].map(lambda specs: content in specs)
    return [
        (FULL_MATCH, sellers[speaks & specialist]),
        (LANGUAGE_ONLY, sellers[speaks & ~specialist]),
    ]


def pick_least_loaded(candidates: pd.DataFrame, load: dict) -> str:
    """Candidate with the lowest load ratio (open leads / capacity) that still has free capacity.
    Ties are broken by seller_id. Returns None if all are full."""
    best_id, best_ratio = None, None
    for seller in candidates.itertuples():
        open_leads = load.get(seller.seller_id, 0)
        if open_leads >= seller.max_open_leads:
            continue
        ratio = open_leads / seller.max_open_leads
        if best_ratio is None or ratio < best_ratio:  # candidates are sorted by seller_id
            best_id, best_ratio = seller.seller_id, ratio
    return best_id


def current_load(existing_assignments: pd.DataFrame) -> dict:
    """{seller_id: number of leads already assigned} from previous runs."""
    if existing_assignments is None or existing_assignments.empty:
        return {}
    return existing_assignments["seller_id"].dropna().value_counts().to_dict()


def assign_leads(leads: pd.DataFrame, sellers: pd.DataFrame, load: dict = None,
                 run_id: str = None) -> pd.DataFrame:
    """Assign each lead to one seller (see the rules above)."""
    load = dict(load or {})
    assigned_at = datetime.now().replace(microsecond=0)
    run_id = run_id or assigned_at.strftime("%Y%m%d%H%M%S")

    rows = []
    for lead in leads.sort_values(["created_at", "lead_id"]).itertuples():
        tiers = candidate_tiers(lead.preferred_language, lead.content_of_interest, sellers)
        n_candidates = sum(len(candidates) for _, candidates in tiers)

        seller_id, match_type = None, UNASSIGNED
        for tier_type, candidates in tiers:
            seller_id = pick_least_loaded(candidates, load)
            if seller_id is not None:
                match_type = tier_type
                load[seller_id] = load.get(seller_id, 0) + 1
                break

        if n_candidates == 0:
            reason = f"no active seller speaks '{lead.preferred_language}'"
        elif match_type == UNASSIGNED:
            reason = f"all {n_candidates} candidate sellers are at full capacity"
        elif match_type == LANGUAGE_ONLY and pd.isna(lead.content_of_interest):
            reason = "lead has no valid content_of_interest"
        elif match_type == LANGUAGE_ONLY:
            reason = f"no available specialist in '{lead.content_of_interest}'"
        else:
            reason = None

        rows.append({
            "lead_id": lead.lead_id,
            "seller_id": seller_id,
            "match_type": match_type,
            "reason": reason,
            "preferred_language": lead.preferred_language,
            "content_of_interest": lead.content_of_interest,
            "assigned_at": assigned_at,
            "run_id": run_id,
        })

    return pd.DataFrame(rows, columns=OUTPUT_COLUMNS)


def summarize(assignments: pd.DataFrame) -> pd.DataFrame:
    """Leads per seller and match type."""
    return (assignments
            .assign(seller_id=assignments["seller_id"].fillna("-"))
            .pivot_table(index="seller_id", columns="match_type", values="lead_id",
                         aggfunc="count", fill_value=0)
            .reindex(columns=[FULL_MATCH, LANGUAGE_ONLY, UNASSIGNED], fill_value=0))

## 6. Read inputs

In [ ]:
client = get_client(creds)

leads = read_table(client, LEADS_TABLE)
sellers = normalize_sellers(read_table(client, SELLERS_TABLE))

print(f"{len(leads)} leads | {len(sellers)} active sellers")
sellers

## 7. Keep only the new leads

A lead is *new* if it is not in the output table yet. With `FULL_REFRESH = True` every lead is reassigned.

In [ ]:
existing = None
if not FULL_REFRESH and table_exists(client, OUTPUT_TABLE):
    existing = read_table(client, OUTPUT_TABLE)
    leads = leads[~leads["lead_id"].isin(existing["lead_id"])]

load = current_load(existing)   # {seller_id: leads already assigned}
print(f"{len(leads)} new leads | {0 if existing is None else len(existing)} previous assignments")
load

## 8. Assign

In [ ]:
assignments = assign_leads(leads, sellers, load=load)
summarize(assignments)

In [ ]:
# Leads that did not get a perfect match, and why
assignments[assignments["match_type"] != "full_match"]

## 9. Materialize

In [ ]:
if assignments.empty:
    print("Nothing to assign.")
elif DRY_RUN:
    print("Dry run: nothing written.")
else:
    write_table(assignments, client, OUTPUT_TABLE, if_exists="replace" if FULL_REFRESH else "append")
    print(f"{len(assignments)} rows written to {OUTPUT_TABLE}")

## 10. Check the output table

In [ ]:
client.query_df(f"""
    SELECT coalesce(seller_id, '-') AS seller_id, match_type, count(*) AS n_leads
    FROM {OUTPUT_TABLE}
    GROUP BY 1, 2
    ORDER BY 1, 2
""")